In [ ]:
    class CitationsETL(SetUp):

    def __init__(self):
        super().__init__()
        return
    
    def citation_count(self):
        sql = """
            CREATE OR REPLACE TABLE memory.citation_count AS  
            -- ETL FOR citation_count
            -- ======================
            SELECT cited_id AS work_id,
                    cited_by_count,
                    count(citer_id) AS cited_by_count_endogenous
            FROM project.citer_cited
            LEFT JOIN project.raw
            ON id = cited_id
            -- WHERE len(authorships) > 0
            GROUP BY ALL
            ORDER BY cited_by_count_endogenous DESC           
            """
        self.db.sql(sql) #.show()
        self.db.sql("SELECT * FROM memory.citation_count").show()
        return

    
    def fwci(self):
        sql = """
            CREATE OR REPLACE TABLE memory.fwci AS
            WITH
            get_fwci_endogenous_CTE AS
                (SELECT DISTINCT cited_id,
                                count(citer_id) OVER (PARTITION BY cited_id)/
                                    (count(citer_id) OVER (PARTITION BY cited_year)/
                                    count(DISTINCT citer_id) OVER (PARTITION BY cited_year)) AS fwci_endogenous,
                FROM project.citer_cited
                ORDER BY fwci_endogenous DESC
                )
            SELECT id AS work_id,
                    fwci,
                    fwci_endogenous
                FROM project.raw
                LEFT JOIN get_fwci_endogenous_CTE
                ON id = cited_id
            ORDER BY fwci DESC
            """
        self.db.sql(sql) #.show()
        self.db.sql("SELECT * FROM memory.fwci").show()
        return
    
    def highly_cited(self):
        sql = """
            CREATE OR REPLACE TABLE memory.highly_cited AS  
            -- ETL FOR highly_cited_work
            -- =========================
            WITH
            citation_counts_CTE AS
                (SELECT DISTINCT cited_by_count,
                        count(citer_id) OVER (PARTITION BY cited_id) AS cited_by_count_endogenous,
                        cited_id,
                        cited_year,
                FROM project.citer_cited
                LEFT JOIN project.raw
                ON id = cited_id
                -- ORDER BY cited_by_count_endogenous DESC
                ),
            quantiles_CTE AS
                (SELECT DISTINCT cited_id,
                        cited_year,
                        quantile_disc(cited_by_count, 0.99) OVER (PARTITION BY cited_year) AS quant,
                        quantile_disc(cited_by_count_endogenous, 0.99) OVER (PARTITION BY cited_year) AS quant_endogenous
                FROM citation_counts_CTE
                -- ORDER BY cited_year DESC
                ),
            highly_cited_CTE AS
                (SELECT DISTINCT cc.cited_id,
                        IF (cited_by_count_endogenous - quant_endogenous > 0, true, false) AS isHighlyCitedEndogenous,
                        IF (cited_by_count - quant > 0, true, false) AS isHighlyCited
                FROM citation_counts_CTE cc
                LEFT JOIN quantiles_CTE
                USING (cited_year)
                )
            SELECT cited_id AS work_id,
                    isHighlyCited,  
                    isHighlyCitedEndogenous
            FROM highly_cited_CTE

            """
        self.db.sql(sql) #.show()
        self.db.sql("SELECT * FROM memory.highly_cited").show()
        return

    
    def disruption_index(self):
        sql = """
            CREATE OR REPLACE TABLE memory.disruption_index AS  
            -- ETL FOR disruption_index
            -- ========================
            WITH
            extract_work_CTE AS
                (SELECT id AS work_id,
                        fwci,
                        unnest(referenced_works) AS referenced_work
                FROM project.raw
                ),
            extract_disruption_index_CTE AS
                (SELECT work_id,
                        100*r1.fwci/avg(r2.fwci) OVER (PARTITION BY work_id) AS disruption_index
                FROM extract_work_CTE r1
                LEFT JOIN project.raw r2
                ON referenced_work = r2.id
                ORDER BY disruption_index DESC
                )

            SELECT DISTINCT *
            FROM extract_disruption_index_CTE
            WHERE disruption_index NOT NULL AND disruption_index != 'NaN' AND disruption_index != 'Infinity'
            ORDER BY disruption_index DESC
            """
        self.db.sql(sql) #.show()
        self.db.sql("SELECT * FROM memory.disruption_index").show()
        return

    
    def topic_indicators(self):
        sql = """
            CREATE OR REPLACE TABLE memory.topic_indicators AS  
            -- ETL FOR topic_indicators
            -- =======================
            WITH
                topics_CTE AS
                (SELECT work_id,
                        domain_id[-1:] AS domain_id,
                        field_id[-2:] AS field_id,
                        subfield_id[-4:] AS subfield_id
                    FROM project.topics
                ),
                citer_cited_topics_CTE AS
                (SELECT citer_id,
                        t1.domain_id AS domain_id_citer,
                        t1.field_id AS field_id_citer,
                        t1.subfield_id AS subfield_id_citer,
                        cited_id,
                        t2.domain_id AS domain_id_cited,
                        t2.field_id AS field_id_cited,
                        t2.subfield_id AS subfield_id_cited,
                    FROM project.citer_cited
                    LEFT JOIN topics_CTE as t1
                    ON citer_id = t1.work_id
                    LEFT JOIN topics_CTE t2
                    ON cited_id = t2.work_id
                ),
                cited_subfield_list_CTE AS
                (SELECT citer_id,
                        list(subfield_id_cited) AS cited_subfield_list
                    FROM citer_cited_topics_CTE
                    GROUP BY citer_id
                ),
                citer_subfield_list_CTE AS
                (SELECT cited_id,
                        list(subfield_id_citer) AS citer_subfield_list
                    FROM citer_cited_topics_CTE
                    GROUP BY cited_id
                ),
                citer_topic_CTE AS
                (SELECT citer_id,
                        subfield_id_citer,
                        list_distinct(cited_subfield_list) AS cited_topics,
                        len(list_distinct(cited_subfield_list)) AS cited_topics_count
                    FROM citer_cited_topics_CTE cct
                    LEFT JOIN cited_subfield_list_CTE
                    USING (citer_id)
                    GROUP BY ALL
                    ORDER BY cited_topics_count DESC
                ),
                cited_topic_CTE AS
                (SELECT cited_id,
                        subfield_id_cited,
                        list_distinct(citer_subfield_list) AS citer_topics,
                        len(list_distinct(citer_subfield_list)) AS citer_topics_count
                    FROM citer_cited_topics_CTE cct
                    LEFT JOIN citer_subfield_list_CTE
                    USING (cited_id)
                    GROUP BY ALL
                    ORDER BY citer_topics_count DESC
                )

            SELECT citer_id AS work_id,
                    subfield_id_citer AS subfield_id,
                    cited_topics,
                    cited_topics_count,
                    citer_topics,
                    citer_topics_count
            FROM citer_topic_CTE
            LEFT JOIN cited_topic_CTE
            ON citer_id = cited_id
            ORDER BY citer_topics_count DESC
            """
        self.db.sql(sql) #.show()
        self.db.sql("SELECT * FROM memory.topic_indicators").show()
        return
    
    def spectral_ranks(self):
        sql = """
            CREATE OR REPLACE TABLE memory.spectral_ranks AS
            -- ETL FOR pagerank AND influence
            -- ==============================
            WITH
                select_work_journal_CTE AS
                (SELECT id AS work_id,
                        "primary_location.source".id AS source_id,
                        pagerank,
                        influence,
                    FROM project.raw
                    LEFT JOIN project.pagerank_source
                    ON "primary_location.source".id = citer
                ),
                select_work_institutions_CTE AS
                (SELECT DISTINCT work_id,
                        institution_id,
                        count(DISTINCT institution_id) OVER (PARTITION BY work_id) AS institution_count,
                        pagerank,
                        influence
                    FROM project.authorships
                    LEFT JOIN project.pagerank_institution
                    ON institution_id = citer 
                ),
                select_work_institution_CTE AS
                (SELECT work_id,
                        sum(pagerank)/institution_count AS pagerank_institution,
                        sum(influence)/institution_count AS influence_institution
                    FROM select_work_institutions_CTE
                    GROUP BY work_id, institution_count
                )

            SELECT work_id,
                    100*pagerank AS pagerank_source,
                    100*influence AS influence_source,
                    100*pagerank_institution AS pagerank_institution,
                    100*influence_institution AS influence_institution
            FROM select_work_journal_CTE
            LEFT JOIN select_work_institution_CTE
            USING (work_id)
            ORDER BY pagerank_institution DESC, pagerank_source DESC
            """
        self.db.sql(sql) #.show()
        self.db.sql("SELECT * FROM memory.spectral_ranks").show()
        return

    def citation_combiner(self):
        self.db.sql("SHOW ALL TABLES").show()
        sql = "CREATE OR REPLACE TABLE project.work_summary AS (SELECT * FROM memory.citation_count\n"
        for tab in ['references_per_page', 'fwci', 'highly_cited', 'copied_references', 
                    'disruption_index', 'self_references', 'topic_indicators', 'spectral_ranks']:
            sql = f"{sql}LEFT JOIN (SELECT * FROM memory.{tab}) USING (work_id)\n"
        sql = f"{sql})"
        print(f'{sql = }')
        self.db.sql(sql)
        df = self.db.sql("SELECT * FROM project.work_summary").df()
        print(f'{df.shape = }\n{df.head(16)}')
        return
            


In [ ]:
def citations_per_work(self):
        sql = """ 
        CREATE OR REPLACE TABLE memory.citations_per_work AS
            SELECT count(work_id) AS cited_by_count_endogenous,
                    sum(cited_by_count) AS cited_by_count_total,
                    author_id,
                    author_name,
                    publication_year
            FROM
                (SELECT DISTINCT id AS work_id,
                    w.cited_by_count,
                    unnest(authorships).author.id AS author_id,
                    unnest(authorships).author.display_name as author_name,
                    w.publication_year
                FROM project.raw w
                LEFT JOIN (SELECT id AS work_id,
                            unnest(referenced_works) AS cited_id
                            FROM project.raw           
                            ) c
                ON w.id = c.cited_id
                )
                GROUP BY author_id, author_name, publication_year
            ORDER BY cited_by_count_endogenous DESC
        """
        self.db.sql(sql)
        sql.db.sql("SELECT * FROM memory_citations_per_work").show()
        return

    def citations_per_work_ranked(self):
        sql = """ 
                CREATE OR REPLACE TABLE memory.citations_per_work_ranked AS
                SELECT cited_id,
                        publication_year,
                        cited_by_count_total,
                        cited_by_count_endogenous,
                        percent_rank(ORDER BY cited_by_count_total) OVER w AS percent_rank_total,
                        percent_rank(ORDER BY cited_by_count_endogenous) OVER w AS percent_rank_endogenous
                FROM memory.citations_per_work 
                WINDOW w AS (PARTITION BY publication_year) -- ORDER BY cited_by_count_total, cited_by_count_endogenous) 
                ORDER BY publication_year DESC, percent_rank_total DESC
                """
        self.db.sql(sql)
        return

    def citation_summation(self):
        sql = """ 
                CREATE OR REPLACE TABLE memory.citations AS
                SELECT author_id,
                        author_name,
                        sum(cited_by_count_total) AS citations_total,
                        sum(cited_by_count_endogenous) AS citations_endogenous
                FROM memory.citations_per_work_ranked m
                LEFT JOIN authorships a
                ON m.cited_id = a.work_id
                WHERE author_id NOT NULL
                GROUP BY author_id, author_name
                ORDER BY citations_endogenous DESC                    "biblio.last_page"
                """
        self.db.sql(sql)
        return

    def hca_summation(self):       

        sql = """ 
                CREATE OR REPLACE TABLE memory.hca_endogenous AS
                SELECT author_id,
                        author_name,
                        count(cited_id) AS hca_endogenous                    "biblio.last_page"
                FROM memory.citations_per_work_ranked m
                LEFT JOIN authorships a
                ON m.cited_id = a.work_id
                WHERE a.work_id NOT NULL 
                        AND percent_rank_endogenous >= 0.99
                GROUP BY ALL
                ORDER BY hca_endogenous DESC;

                CREATE OR REPLACE TABLE memory.hca_total AS
                SELECT author_id,
                        author_name,
                        count(cited_id) AS hca_total,
                FROM memory.citations_per_work_ranked m
                LEFT JOIN authorships a
                ON m.cited_id = a.work_id
                WHERE a.work_id NOT NULL 
                        AND percent_rank_total >= 0.99
                GROUP BY ALL
                ORDER BY hca_total DESC
                """
        self.db.sql(sql)
        return
    
    def citations_endogenous_all(self):
        sql = """ 
            CREATE OR REPLACE TABLE memory.citations_endogenous_all AS
                SELECT author_id,
                        author_name,
                        citations_total,
                        citations_endogenous,
                        hca_total,
                        hca_endogenous
                FROM memory.citations
                LEFT JOIN
                    (SELECT t.*,
                            e.hca_endogenous
                        FROM memory.hca_total t
                        LEFT JOIN memory.hca_endogenous e
                        USING (author_id)
                    ) sub
                USING (author_id, author_name)
                ORDER BY citations_endogenous DESC
            """
        self.db.sql(sql)
        return

    def author_works_count(self):
        sql = """
            CREATE OR REPLACE TABLE memory.author_works_counts AS
                SELECT au.author_id,
                        au.author_name,
                        a.first,
                        a.middle,
                        a.last,
                        a.fullname,
                        a.orcid,
                        a.display_name_alternatives,
                        count(work_id) AS works_count_endogenous,
                        works_count,
                        cited_by_count,
                        "2yr_mean_citedness",
                        h_index      
                    FROM econ.authorships au
                        LEFT JOIN econ.authors a
                        ON a.author_id = au.author_id
                    GROUP BY ALL
                    ORDER BY works_count_endogenous DESC
            """
        self.db.sql(sql)
        return
    
    def citation_summary(self):

        sql = """ 
            CREATE OR REPLACE TABLE econ.citation_summary AS
                SELECT DISTINCT c.author_id,
                        c.author_name,
                        a.author_name,
                        a.works_count_endogenous,
                        s.citations_total AS citations_total_,
                        c.citations_endogenous,
                        hca_total,
                        hca_endogenous,
                        a.orcid,
                        a.display_name_alternatives,
                        a.works_count AS works_count_total,
                        a.cited_by_count,
                        a."2yr_mean_citedness",                
                SetUp:

    def __init__(self):
        self._setup_db()
        return

        self.db.sql("SHOW ALL TABLES").show()
        return
                        a.h_index
                FROM memory.citations c
                    LEFT JOIN memory.citations_endogenous_all s
                    ON c.author_id = s.author_id
                        LEFT JOIN memory.author_works_counts a
                        ON c.author_id = a.author_id
            ORDER BY cited_by_count DESC, h_index DESC
            """
        self.db.sql(sql)
        return
    
    def show_all(self):
        self.db.sql("SELECT * FROM memory.citations_per_work").show()
        self.db.sql("SELECT * FROM memory.citations_per_work_ranked").show()
        self.db.sql("SELECT * FROM memory.citations").show()
        self.db.sql("SELECT * FROM memory.hca_endogenous").show() 
        self.db.sql("SELECT * FROM memory.citations_endogenous_all").show()
        self.db.sql("SELECT * FROM econ.authors").show()  
        self.db.sql("SELECT * FROM econ.citation_summary").show()
        return
    
    def load_citations(self):
        df = self.db.sql("""
                         SELECT * EXCLUDE (author_name_1, display_name_alternatives) FROM project.citation_summary ORDER BY hca_endogenous DESC, h_index DESC
                         """).df().reset_index(drop=True)
        df.to_excel('../DATA/citation_summary.xlsx', index=False)
        return
